# GAP score — versione definitiva

Questo è l'**unico** GAP score del progetto: una sola formula, con ogni scelta
motivata. Il notebook è autosufficiente (si esegue da solo) e produce un solo
file di output. La giustificazione dettagliata delle due correzioni è nel
notebook `correzione_flotte_aci` (lato domanda) e nella sezione **«Evidenza»**
di questo stesso notebook (lato offerta); l'audit di coerenza dell'intera
pipeline è in `audit_coerenza_metodologica`. Questo è la versione di produzione.

## La formula

$$\boxed{\;\text{GAP}_i = \text{rank\%}(D_i)\; -\; \text{rank\%}(S_i)\;}$$

dove, per ogni sezione di censimento $i$ dell'universo eleggibile:

- $D_i$ = **domanda** effettiva di ricarica pubblica;
- $S_i$ = **accessibilità dell'offerta** di ricarica;
- $\text{rank\%}$ = rango percentile nazionale (calcolato solo sulle sezioni
  eleggibili).

GAP vicino a **+1** = deserto (tanta domanda, offerta minima); vicino a **−1** =
sezione ben servita; vicino a **0** = equilibrata.

Le tre grandezze di ingresso e il perché di ciascuna scelta sono nella sezione
successiva.

## Giustificazione di ogni scelta

### Domanda $D_i$ — costruita in tre passi

| Passo | Cosa | Perché questa scelta (e non l'alternativa) |
|---|---|---|
| 1. Base | `veicoli_da_ricaricare_stimati`: EV + ibridi plug-in 2025, disaggregati dal totale provinciale con peso `popolazione_età_guida × moltiplicatore IDI3` (β=0,50) | La sola popolazione tratterebbe un quartiere popolare come uno di ville a pari abitanti. L'IDI3 differenzia per benessere relativo; β=0,50 è calibrato su un dato reale (immatricolazioni ACI LOD 2024 per comune). *(pipeline del gruppo)* |
| 2. Correzione flotte | Il totale provinciale è deflazionato dalla quota di immatricolazioni di flotta e ridistribuito sulla popolazione | Il totale ACI è per provincia di *immatricolazione*: le flotte di noleggio gonfiano Trento (89% di EV in flotta nel 2019), Firenze, Bolzano. Senza correzione, Trento pesa il 6,6% della domanda nazionale con lo 0,8% della popolazione, e i "deserti peggiori" sono un artefatto contabile. *(→ `correzione_flotte_aci`)* |
| 3. Fattibilità domestica | Domanda × `rank%(E27/E3)` (interni per edificio residenziale) | Chi ha un box privato (villetta) non dipende dalla ricarica pubblica come chi vive in condominio. `E27/E3` alto = condominio = più bisogno pubblico. È la correzione più "soft" (proxy non validata empiricamente), ma concettualmente fondata. *(pipeline del gruppo)* |

### Offerta $S_i$ — accessibilità *distance-aware*

L'offerta di partenza (numero di colonnine entro 500 m) ha un difetto grave: il
**47,5% delle sezioni ha zero colonnine** e riceve tutta lo stesso identico
punteggio. Poiché i deserti stanno tutti in quel blocco, il GAP di base li
ordina di fatto **solo per domanda**, senza distinguere una sezione con una
colonnina a 500 m da una isolata a 15 km — pur avendo il progetto già calcolato
quella distanza. La definizione adottata qui recupera quel dato:

- sezioni **servite** (≥ 1 colonnina entro 500 m): ordinate per **numero** di
  colonnine (la densità conta: 50 colonnine raggiungibili ≠ 1);
- sezioni **non servite** (0 entro 500 m): ordinate per **prossimità** della
  colonnina più vicina, tutte collocate *sotto* le servite.

In sintesi: «quante colonnine posso raggiungere a piedi; se nessuna, quanto è
lontana la più vicina». *(evidenza quantitativa nella sezione «Evidenza» più sotto)*

### Normalizzazione e differenza

`rank%` (rango percentile nazionale) invece di min-max: rende confrontabili
domanda (veicoli) e offerta (colonnine), è robusto agli outlier (una sezione
fuori scala vale «la più alta», non trascina la distribuzione) ed è già la
trasformazione usata per l'IDI. La **differenza** (non il rapporto) evita di
esplodere quando l'offerta è zero e produce un punteggio limitato in $[-1,+1]$,
direttamente leggibile.

### Universo di calcolo

Solo le sezioni **eleggibili** (`flag_eleggibile_EV`: abitate, non speciali,
con popolazione in età di guida > 0) e con offerta calcolata. I rank sono
calcolati **solo** su queste, altrimenti le sezioni escluse si accumulerebbero
al rango più basso e comprimerebbero la scala. Alle altre il GAP resta `NaN`
(«nessun dato», mai zero).

In [ ]:
import io, csv, json, re, unicodedata
from pathlib import Path
import numpy as np
import pandas as pd

# Percorsi risolti rispetto alla radice del repository (il notebook gira da
# notebooks/): gli input si cercano prima in output/ (rigenerati) e poi in
# data/ o config/ - la stessa convenzione di src/paths.py.
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT_DIR = REPO / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

def trova(nome, *cartelle):
    """Primo percorso esistente fra `cartelle`, altrimenti il primo candidato."""
    for cartella in cartelle:
        if (cartella / nome).exists():
            return cartella / nome
    return cartelle[0] / nome

# Il passo 9 (05_merge_auto_colonnine_geo.py) produce
# `sezioni_offerta_domanda_merged.parquet`: e' lo stesso file che qui si
# chiamava `sezioni_gap_score.parquet` e che andava rinominato a mano.
# Accettando entrambi i nomi, quel passaggio manuale non serve piu'.
BASE = next(
    (p for p in (trova("sezioni_offerta_domanda_merged.parquet", OUT_DIR, REPO / "data"),
                 trova("sezioni_gap_score.parquet", OUT_DIR, REPO / "data"))
     if p.exists()),
    OUT_DIR / "sezioni_offerta_domanda_merged.parquet",
)
CACHE_FLOTTE = trova("quota_flotte_ev_2019_per_provincia.json", OUT_DIR, REPO / "config")
OUT_PARQUET = OUT_DIR / "sezioni_gap_score_DEFINITIVO.parquet"
OUT_CSV = OUT_DIR / "gap_score_definitivo.csv"

def norm(s):
    s = str(s).upper().strip()
    s = unicodedata.normalize("NFKD", s).encode("ascii","ignore").decode("ascii")
    s = s.replace("'"," ").replace("-"," ").replace("/"," ")
    return re.sub(r"\s+"," ", s).strip()

df = pd.read_parquet(BASE)
for c in ["flag_eleggibile_EV"]:
    df[c] = df[c].map({True:True, False:False}).fillna(False).astype(bool)
# conservo il punteggio ORIGINALE (domanda non corretta + offerta a solo conteggio),
# presente nel file base, solo per mostrare più avanti l'effetto delle due correzioni.
gap_originale = df["gap_score"].copy()
offerta_norm_originale = df["offerta_norm"].copy()
print(f"Sezioni caricate: {len(df):,} | colonne: {df.shape[1]}")
el = df.flag_eleggibile_EV & df.offerta_colonnine_500m.notna()   # universo di calcolo del GAP
print(f"Universo eleggibile per il GAP: {el.sum():,}")


## Passo 1-2 — Domanda corretta dal bias flotte

Applichiamo la correzione flotte al totale provinciale (quantità 2025,
composizione d'uso 2019) e la ri-disaggregiamo alle sezioni con gli **stessi
pesi IDI×popolazione** già calcolati. La giustificazione dettagliata e i dati
d'uso ACI sono in `correzione_flotte_aci`; qui la applichiamo in forma compatta.

In [ ]:
# quota-flotta 2019 per provincia (prodotta da correzione_flotte_aci)
quote = json.load(open(CACHE_FLOTTE, encoding="utf-8"))
ALIAS = {"BARLETTA ANDRIA TRANI":"BARLETTA TRANI","BOLZANO BOZEN":"BOLZANO","FORLI":"FORLI CESENA",
         "MONZA E DELLA BRIANZA":"MONZA BRIANZA","PESARO":"PESARO E URBINO","REGGIO DI CALABRIA":"REGGIO CALABRIA",
         "REGGIO NELL EMILIA":"REGGIO EMILIA","VERBANIA":"VERBANO CUSIO OSSOLA"}
key = lambda x: ALIAS.get(norm(x), norm(x))
qf   = {key(k): v["quota_flotta_2019"] for k,v in quote.items()}
ev19 = {key(k): v["ev_2019"]           for k,v in quote.items()}
naz  = sum(v["ev_flotta_2019"] for v in quote.values()) / sum(v["ev_2019"] for v in quote.values())

# totale provinciale originale + popolazione in età di guida (dal file base)
g = df[df.flag_eleggibile_EV].groupby("PROVINCIA_EV_2025")
prov = g.agg(ev=("EV_PROVINCIALE","first"),
             popg=("popolazione_eta_guida_stimata","sum")).reset_index()
prov["k"] = prov.PROVINCIA_EV_2025.map(norm)
prov["ev19"] = prov.k.map(ev19)
prov["f"] = np.where((prov.ev19 >= 200) & prov.k.map(qf).notna(), prov.k.map(qf), naz)
# deflazione + redistribuzione del bacino flotte proporzionale alla popolazione
prov["privati"] = prov.ev * (1 - prov.f)
pool = (prov.ev * prov.f).sum()
prov["ev_corretto"] = prov.privati + pool * prov.popg / prov.popg.sum()
assert abs(prov.ev_corretto.sum() - prov.ev.sum()) < 1.0
tot_corr = prov.set_index("PROVINCIA_EV_2025")["ev_corretto"]
print(f"Totale nazionale conservato: {prov.ev.sum():,.0f} -> {prov.ev_corretto.sum():,.0f}")

# ri-disaggregazione a sezione con gli stessi pesi/cap della pipeline
def alloca_con_cap(pesi, cap, totale, tol=1e-9):
    pesi=np.asarray(pesi,float); cap=np.asarray(cap,float); totale=float(totale)
    if totale<=tol: return np.zeros_like(pesi)
    if cap.sum()+tol<totale: raise ValueError("totale > capacita'")
    alloc=np.zeros_like(pesi); res=totale; att=(pesi>0)&(cap>0)
    while res>tol:
        capres=cap-alloc; cand=att&(capres>tol)
        if not cand.any(): raise RuntimeError("capacita' esaurita")
        pc=pesi[cand]
        if pc.sum()<=tol: pc=capres[cand]
        prop=res*pc/pc.sum(); oltre=prop>capres[cand]+tol; pos=np.flatnonzero(cand)
        if not oltre.any(): alloc[pos]+=prop; res=0.0
        else:
            pcap=pos[oltre]; a=capres[pcap]; alloc[pcap]+=a; res-=a.sum(); att[pcap]=False
    return alloc

df["domanda_corretta"] = 0.0
for p, idx in df.groupby("PROVINCIA_EV_2025").groups.items():
    if p not in tot_corr.index: continue
    idx=list(idx)
    df.loc[idx,"domanda_corretta"] = alloca_con_cap(
        df.loc[idx,"peso_EV"].to_numpy(), df.loc[idx,"cap_EV_sezione"].to_numpy(), float(tot_corr[p]))
sc = (df.groupby("PROVINCIA_EV_2025").domanda_corretta.sum() - tot_corr.reindex(
       df.groupby("PROVINCIA_EV_2025").domanda_corretta.sum().index)).abs().max()
print(f"Max scarto riconciliazione provinciale: {sc:.2e}")


## Passo 3 — Domanda effettiva (× fattibilità domestica)

In [ ]:
df["domanda_effettiva"] = np.where(el, df.domanda_corretta * df.quota_bisogno_pubblico_sezione, np.nan)
print("domanda_effettiva calcolata sulle sezioni eleggibili.")
print(df.loc[el,"domanda_effettiva"].describe()[["mean","50%","max"]].round(3).to_string())


## Offerta — accessibilità *distance-aware*

In [ ]:
def offerta_accessibilita(offerta, distanza):
    offerta=np.asarray(offerta,float); distanza=np.asarray(distanza,float); n=len(offerta)
    served = offerta > 0
    supply = np.empty(n)
    supply[~served] = pd.Series(-distanza[~served]).rank().to_numpy()          # non servite: piu' vicina = piu' offerta
    supply[served]  = (~served).sum() + pd.Series(offerta[served]).rank().to_numpy()  # servite: sempre sopra, per numero
    return pd.Series(supply).rank(pct=True).to_numpy()

sub = df[el]
off_norm = offerta_accessibilita(sub.offerta_colonnine_500m, sub.distanza_colonnina_piu_vicina_m)
df["offerta_norm"] = np.nan
df.loc[el, "offerta_norm"] = off_norm
print(f"Valori distinti di offerta_norm: {df.loc[el,'offerta_norm'].nunique():,} "
      f"(la versione a solo conteggio ne aveva ~200: il blocco di pareggio e' risolto)")


## GAP score definitivo

In [ ]:
df["domanda_norm"] = np.nan
df.loc[el,"domanda_norm"] = df.loc[el,"domanda_effettiva"].rank(pct=True)
df["gap_score"] = np.nan
df.loc[el,"gap_score"] = df.loc[el,"domanda_norm"] - df.loc[el,"offerta_norm"]

s = df.loc[el,"gap_score"]
print(f"GAP score: n={s.notna().sum():,} | min={s.min():.3f} | mediana={s.median():.3f} | max={s.max():.3f}")
assert s.between(-1,1).all()
assert df.loc[~el,"gap_score"].isna().all()


## Verifiche di solidità

Controlli di buon senso: la direzione del punteggio (GAP alto ⇒ offerta bassa),
e la composizione della classifica dei peggiori deserti, che non deve più essere
dominata dagli artefatti di immatricolazione.

In [ ]:
# direzione
top = df.loc[el].nlargest(2000,"gap_score"); bot = df.loc[el].nsmallest(2000,"gap_score")
print(f"Offerta media: peggiori 2000 deserti = {top.offerta_colonnine_500m.mean():.1f} colonnine | "
      f"migliori 2000 = {bot.offerta_colonnine_500m.mean():.1f}")
assert top.offerta_colonnine_500m.mean() < bot.offerta_colonnine_500m.mean()

# composizione dei peggiori deserti (deve essere plausibile, non tutta Trento)
print("\nProvince più rappresentate tra i 1.000 peggiori deserti:")
print(df.loc[el].nlargest(1000,"gap_score").PROVINCIA.value_counts().head(8).to_string())
print("\nSezioni di Trento tra i 1.000 peggiori deserti:",
      int((df.loc[el].nlargest(1000,"gap_score").PROVINCIA=="Trento").sum()),
      "(erano 183 senza la correzione flotte)")


## Evidenza: perché le due correzioni (offerta *distance-aware* + flotte)

Confrontiamo il punteggio definitivo con quello di partenza (già presente nel
file base: domanda non corretta + offerta a solo conteggio), per mostrare che i
due interventi non sono cosmetici ma cambiano proprio i deserti.

In [ ]:
# 1) il collo di bottiglia dell'offerta a solo conteggio
zero = el & (df.offerta_colonnine_500m == 0)
print(f"Sezioni a offerta 0 (nessuna colonnina entro 500 m): {zero.sum():,} = {zero.sum()/el.sum():.0%} delle eleggibili")
print(f"  offerta a solo conteggio: {offerta_norm_originale[el].nunique()} valori distinti -> tutte le 0-colonnine pareggiano")
print(f"  offerta distance-aware:   {df.loc[el,'offerta_norm'].nunique():,} valori distinti -> il pareggio e' risolto")

# 2) due sezioni ad alta domanda e 0 colonnine, ma isolamento opposto
alta = el & (df.offerta_colonnine_500m == 0) & (df.domanda_norm > 0.9)
vic  = df.loc[alta].loc[df.loc[alta,"distanza_colonnina_piu_vicina_m"].idxmin()]
iso  = df.loc[alta].loc[df.loc[alta,"distanza_colonnina_piu_vicina_m"].idxmax()]
print("\nDue sezioni ad alta domanda, 0 colonnine, isolamento opposto:")
print(f"  quasi servita  {vic.COMUNE:<14} colonnina a {vic.distanza_colonnina_piu_vicina_m:6.0f} m -> gap definitivo {vic.gap_score:.3f}")
print(f"  isolata        {iso.COMUNE:<14} colonnina a {iso.distanza_colonnina_piu_vicina_m:6.0f} m -> gap definitivo {iso.gap_score:.3f}")
print("  (a solo conteggio avrebbero avuto lo STESSO punteggio di offerta)")

# 3) effetto flotte sulla classifica dei deserti
tn_orig = int((df.loc[el].assign(g=gap_originale[el]).nlargest(1000,"g").PROVINCIA=="Trento").sum())
tn_def  = int((df.loc[el].nlargest(1000,"gap_score").PROVINCIA=="Trento").sum())
print(f"\nSezioni di Trento tra i 1.000 peggiori deserti: {tn_orig} (punteggio originale) -> {tn_def} (definitivo)")


## Salvataggio — un solo file

Salviamo il file completo (con geometria) con **un'unica** colonna `gap_score`.
Le colonne `domanda_norm` / `offerta_norm` / `domanda_effettiva` /
`domanda_corretta` sono conservate solo come componenti tracciabili del
punteggio, non come punteggi alternativi.

In [ ]:
# rimuovo eventuali residui di versioni precedenti del punteggio, tengo una sola gap_score
drop_cols = [c for c in ["domanda_effettiva_sezione"] if c in df.columns]
out = df.drop(columns=drop_cols)
out.to_parquet(OUT_PARQUET, index=False)
out[["SEZ2011","gap_score"]].to_csv(OUT_CSV, index=False)
print(f"Salvati (un solo GAP score):")
print(f"  {OUT_PARQUET.name}  ({len(out):,} righe, con geometria)")
print(f"  {OUT_CSV.name}  (SEZ2011 + gap_score)")


## Limiti dichiarati (validi per questo punteggio)

- La quota-flotta usata per la correzione della domanda è la composizione d'uso
  **2019** (unico anno ACI con la finalità d'uso), applicata ai totali 2025: è
  una stima prudente (il noleggio è cresciuto dopo il 2019). **Aosta** resta
  sovrastimata (inflazione da km0, non da flotte).
- `E27/E3` come proxy di fattibilità di ricarica domestica non è validato
  empiricamente: è la componente più debole della domanda.
- Il buffer di 500 m e la quota plug-in UNRAE (27,6%) sono convenzioni non
  calibrate su dati locali.
- La distanza dalla colonnina più vicina è disponibile solo per le sezioni a
  offerta 0 (per le servite è < 500 m per definizione): serve esattamente lì,
  a ordinare il blocco dei non serviti.
- Tutte le variabili censuarie (popolazione, IDI, edifici) sono del **2011**.

In [ ]:
n_milano = df.loc[df.PROVINCIA.str.upper() == "MILANO", "gap_score"].notna().sum()
print(f"Sezioni con gap_score calcolato in provincia di Milano: {n_milano:,}")